# CS156 Session 7 — Networks 1: Feed-Forward Neural Networks

This notebook explores how feed-forward neural networks process information and learn from data. I first examine a neural network built with TensorFlow/Keras, then connect the code to its mathematical forward pass. I also examine a perceptron built from scratch and adapt it to classify images from FashionMNIST.

The main ideas I focus on are layers, units, weights, biases, activation functions, forward propagation, and parameter learning.

# Question 1 — Basic Questions for Class: Describe the Neural Network

## My Interpretation

This network is learning to classify handwritten digits from the MNIST dataset. Each image is $28 \times 28$ pixels, so the image contains 784 grayscale values. The pixels are normalized from the original 0–255 range to 0–1 before entering the network.

The network has four Keras layers:

- **Flatten:** reshaping the $28 \times 28$ image into 784 values.
- **Dense(128):** processing the input with 128 sigmoid units.
- **Dense(64):** combining the previous activations with 64 sigmoid units.
- **Dense(10):** producing 10 output values, one for each possible digit.

A **unit**, or artificial neuron, computes a weighted sum of its inputs, adds a bias, and then usually applies an activation function:

$$
z = \mathbf{w}^T\mathbf{x} + b
$$

$$
a = \phi(z)
$$

The Dense layers are fully connected; every unit receives information from every unit in the previous layer. The weights determine how strongly each input influences the next unit, while the bias shifts when that unit becomes active.

The two hidden layers use the sigmoid activation:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

The final `Dense(10)` layer does not specify an activation function, so it produces 10 raw output scores.

One limitation of the provided code is that `BinaryCrossentropy()` normally expects probability-like outputs, while the final layer produces unrestricted values. The validation data are also not normalized or one-hot encoded in the original code. I therefore interpret this section mainly as demonstrating the structure and operation of a feed-forward network rather than as an ideal MNIST implementation.

In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.losses import CategoricalCrossentropy

# Loading the MNIST data
(x_train, y_train), (x_val, y_val) = mnist.load_data()

# Normalizing the pixel values to be between 0 and 1
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0

# Converting the digit labels into one-hot vectors
y_train = to_categorical(y_train, 10)
y_val = to_categorical(y_val, 10)

# Building the feed-forward neural network
model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28)), # Standard way to define input shape in newer Keras
    Flatten(),  # Flattening each image into 784 values
    Dense(128, activation="sigmoid"),  # Computing the first hidden layer
    Dense(64, activation="sigmoid"),  # Computing the second hidden layer
    Dense(10, activation="softmax")  # Standard output activation for multi-class
])

# Preparing the model for training with categorical crossentropy
model.compile(optimizer="SGD", loss=CategoricalCrossentropy(), metrics=["accuracy"])

# Training the model for 10 epochs
model.fit(x_train, y_train, epochs=10)

# Continuing the training while evaluating validation data
model.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val))

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.3685 - loss: 2.2042
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6385 - loss: 1.7283
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7377 - loss: 1.1082
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8047 - loss: 0.7976
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8406 - loss: 0.6337
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8609 - loss: 0.5398
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8735 - loss: 0.4810
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8824 - loss: 0.4409
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8889 - loss: 0.4117
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8930 - loss: 0.3893
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8965 - loss: 0.3717 - val_accuracy: 0.9015 - val_l

# Question 2 — Draw the Network and Write the Forward Pass

## Network Structure

The network can be summarized as:

$$
784 \rightarrow 128 \rightarrow 64 \rightarrow 10
$$

The flattened image provides 784 input values. These values move through two hidden layers and finally produce 10 output scores.

## Forward Pass

For the first hidden layer:

$$
\mathbf{z}^{(1)} = W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}
$$

$$
\mathbf{a}^{(1)} = \sigma(\mathbf{z}^{(1)})
$$

where:

$$
W^{(1)} \in \mathbb{R}^{128 \times 784}
$$

For the second hidden layer:

$$
\mathbf{z}^{(2)} = W^{(2)}\mathbf{a}^{(1)} + \mathbf{b}^{(2)}
$$

$$
\mathbf{a}^{(2)} = \sigma(\mathbf{z}^{(2)})
$$

where:

$$
W^{(2)} \in \mathbb{R}^{64 \times 128}
$$

For the output layer:

$$
\hat{\mathbf{y}} = W^{(3)}\mathbf{a}^{(2)} + \mathbf{b}^{(3)}
$$

where:

$$
W^{(3)} \in \mathbb{R}^{10 \times 64}
$$

Combining everything gives:

$$
\hat{\mathbf{y}}
=
W^{(3)}
\sigma
\left(
W^{(2)}
\sigma
\left(
W^{(1)}\mathbf{x}+\mathbf{b}^{(1)}
\right)
+\mathbf{b}^{(2)}
\right)
+\mathbf{b}^{(3)}
$$

I interpret the forward pass as information moving from the image toward the prediction. Each layer transforms the previous layer using weights and biases; the sigmoid functions add nonlinearity, allowing the network to represent more complicated relationships than a single linear transformation.

In [3]:
# Displaying the network structure and parameter counts
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,388 (427.30 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

# Question 3 — Core Questions: Describe the Perceptron from Scratch

## My Interpretation

The second model is much simpler than the MNIST neural network. It is a **single-layer perceptron**, meaning there are no hidden layers. Every input feature connects directly to one output unit.

The perceptron first computes:

$$
z = b + \sum_{i=1}^{d} w_i x_i
$$

It then applies a step function:

$$
\hat{y} =
\begin{cases}
1, & z \geq 0 \\
0, & z < 0
\end{cases}
$$

The `predict()` function performs this forward calculation for one example.

The `train_weights()` function teaches the perceptron by comparing the prediction with the true label:

$$
e = y - \hat{y}
$$

The bias is updated using:

$$
b \leftarrow b + \eta e
$$

and each feature weight is updated using:

$$
w_i \leftarrow w_i + \eta e x_i
$$

where $\eta$ is the learning rate.

Unlike the deeper MNIST network, this perceptron can only learn a linear decision boundary. However, implementing it from scratch makes the role of the weights, bias, prediction, and parameter updates much easier to see directly.

In [4]:
# Defining the prediction function
def predict(row, weights):
    # Starting the activation with the bias
    activation = weights[0]

    # Computing the weighted sum of the input features
    for i in range(len(row) - 1):
        activation += weights[i + 1] * row[i]

    # Applying the step activation function
    return 1.0 if activation >= 0.0 else 0.0


# Defining the training function
def train_weights(train, l_rate, n_epoch):
    # Creating one bias and one weight for every input feature
    weights = [0.0 for i in range(len(train[0]))]

    # Repeating the learning process across epochs
    for epoch in range(n_epoch):

        # Going through each training example
        for row in train:
            # Making the current prediction
            prediction = predict(row, weights)

            # Computing how wrong the prediction was
            error = row[-1] - prediction

            # Updating the bias
            weights[0] += l_rate * error

            # Updating each feature weight
            for i in range(len(row) - 1):
                weights[i + 1] += l_rate * error * row[i]

    return weights


# Defining the accuracy calculation
def accuracy_metric(actual, predicted):
    correct = 0

    # Comparing each prediction with its correct label
    for i in range(len(actual)):
        if actual[i] == predicted[i]:
            correct += 1

    # Computing the percentage classified correctly
    return correct / len(actual) * 100

# Question 4 — Draw the Perceptron and Write Its Forward Pass

## Network Structure

The perceptron has one direct transformation:

$$
d \text{ input features} \rightarrow 1 \text{ output unit}
$$

There are no hidden layers.

For one example:

$$
\mathbf{x}
=
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots \\
x_d
\end{bmatrix}
$$

and:

$$
\mathbf{w}
=
\begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_d
\end{bmatrix}
$$

The forward pass is:

$$
z = \mathbf{w}^T\mathbf{x} + b
$$

followed by:

$$
\hat{y} =
\begin{cases}
1, & z \geq 0 \\
0, & z < 0
\end{cases}
$$

The weights determine how strongly each feature influences the decision, while the bias shifts the decision boundary.

# Question 5 — Extension: Apply the Perceptron to FashionMNIST

## My Approach and Justification

The original perceptron needs an appropriate input dataset, so I adapted it to FashionMNIST. I selected only **T-shirt/top** and **Trouser** because a single perceptron performs binary classification.

Each FashionMNIST image is $28 \times 28$ pixels. I first flatten each image into 784 values and normalize the pixels to the range 0–1.

This creates a relatively large number of features for a simple perceptron, so I use Principal Component Analysis (PCA) to reduce the input from 784 features to 50 principal components:

$$
784 \text{ pixel features} \rightarrow 50 \text{ PCA features}
$$

PCA is not doing the classification. Instead, it is compressing the images into a smaller set of features that preserve most of their variation. The perceptron then learns a linear boundary between T-shirts/tops and trousers using those 50 features.

This gives the complete pipeline:

$$
28 \times 28
\rightarrow
784
\rightarrow
50
\rightarrow
\text{Perceptron}
\rightarrow
\{0,1\}
$$

In [5]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist
from sklearn.decomposition import PCA

# Defining the prediction function
def predict(row, weights):
    # Starting with the bias
    activation = weights[0]

    # Computing the weighted sum
    for i in range(len(row) - 1):
        activation += weights[i + 1] * row[i]

    # Applying the step function
    return 1.0 if activation >= 0.0 else 0.0


# Defining the training function
def train_weights(train, l_rate, n_epoch):
    # Creating one bias and one weight for each feature
    weights = [0.0 for i in range(len(train[0]))]

    # Repeating the training process
    for epoch in range(n_epoch):

        # Going through each training example
        for row in train:
            # Making a prediction
            prediction = predict(row, weights)

            # Computing the prediction error
            error = row[-1] - prediction

            # Updating the bias
            weights[0] += l_rate * error

            # Updating the feature weights
            for i in range(len(row) - 1):
                weights[i + 1] += l_rate * error * row[i]

    return weights


# Defining the accuracy function
def accuracy_metric(actual, predicted):
    correct = 0

    # Comparing predictions with the correct labels
    for i in range(len(actual)):
        if actual[i] == predicted[i]:
            correct += 1

    # Computing the accuracy percentage
    return correct / len(actual) * 100


# Loading the FashionMNIST dataset
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Keeping T-shirt/top and Trouser images
train_mask = np.isin(y_train, [0, 1])
test_mask = np.isin(y_test, [0, 1])

# Filtering the training examples
x_train = x_train[train_mask]
y_train = y_train[train_mask]

# Filtering the testing examples
x_test = x_test[test_mask]
y_test = y_test[test_mask]

# Flattening and normalizing the images
x_train = x_train.reshape(len(x_train), -1).astype("float32") / 255.0
x_test = x_test.reshape(len(x_test), -1).astype("float32") / 255.0

# Checking the original feature count
print("Original number of features:", x_train.shape[1])

# Creating PCA with 50 components
pca = PCA(n_components=50)

# Learning the PCA transformation from the training data
x_train_pca = pca.fit_transform(x_train)

# Applying the learned transformation to the test data
x_test_pca = pca.transform(x_test)

# Checking the reduced feature count
print("Features after PCA:", x_train_pca.shape[1])

# Measuring how much variance the 50 components are keeping
print("Variance explained:", pca.explained_variance_ratio_.sum())

# Combining the training features and labels
train_data = np.column_stack((x_train_pca, y_train)).tolist()

# Combining the testing features and labels
test_data = np.column_stack((x_test_pca, y_test)).tolist()

# Setting the learning rate
learning_rate = 0.01

# Setting the number of epochs
n_epoch = 30

# Training the perceptron
weights = train_weights(train_data, learning_rate, n_epoch)

# Storing the test predictions
predictions = []

# Predicting each test example
for row in test_data:
    prediction = predict(row, weights)
    predictions.append(prediction)

# Collecting the true test labels
actual = [row[-1] for row in test_data]

# Computing the final classification accuracy
accuracy = accuracy_metric(actual, predictions)

# Displaying the final result
print("Accuracy: %.3f%%" % accuracy)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Original number of features: 784
Features after PCA: 50
Variance explained: 0.8934674
Accuracy: 98.100%


## Results and Interpretation

The original FashionMNIST images contain **784 pixel features**. PCA reduced these to **50 principal components** while retaining about **89.3% of the variance**.

The final perceptron achieved approximately **98.1% accuracy** when distinguishing T-shirts/tops from trousers.

This result makes sense because I intentionally chose two visually different FashionMNIST categories. The experiment also shows that a simple linear classifier can work well when the classification problem is relatively separable and the input has been transformed into a useful lower-dimensional representation.

The most important point is that PCA and the perceptron perform different jobs: PCA is reducing and reorganizing the features, while the perceptron is learning the actual classification boundary.

# Main Takeaway

A feed-forward neural network transforms an input through a sequence of weighted layers. Each neuron computes a weighted sum plus a bias, then an activation function determines the value passed forward.

For a layer:

$$
\mathbf{z}^{(l)} = W^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}
$$

$$
\mathbf{a}^{(l)} = \phi(\mathbf{z}^{(l)})
$$

A perceptron is the simplest version of this idea. It has no hidden layers and applies a threshold directly to a weighted combination of the input features.

The session helped me connect the code to the mathematics: a neural network is not just a collection of Python layers; it is a composition of matrix operations, biases, and nonlinear functions that transform an input into a prediction.